# Flickr30k T3→S3 INT8 Qualitative Audit

Sinh một hình 8 trường hợp Validation để mô tả sự khác biệt giữa ONNX FP32 và
mixed INT8: hai trường hợp tăng CIDEr-D lớn nhất, hai trường hợp giảm lớn nhất,
hai caption thay đổi gần delta 0 nhất và hai đối chứng caption giống hệt.

Đây là phân tích mô tả sau kiểm định, không phải endpoint xác nhận và không
được dùng để thay đổi phương pháp hay ngưỡng.

## Kaggle input bắt buộc — đúng 3 input

1. Output `F30K_T3S3_INT8_Validation`.
2. Output `F30K_T3S3_INT8_NonInferiority`.
3. Dataset `Flickr Image dataset`.

Runtime: CPU / Accelerator None / Internet Off, dự kiến 2–5 phút.

## Cell 1 — Inputs and integrity

In [1]:
import hashlib
import json
import os
import textwrap
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

EXPERIMENT_ID = "F30K_T3S3_INT8_Qualitative"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

VALIDATION_RESULT_PATH = unique_named(
    "F30K_T3S3_INT8_Validation_RESULT.json"
)
CAPTIONS_PATH = unique_named(
    "F30K_T3S3_INT8_Validation_CAPTIONS.csv"
)
NONINFERIORITY_RESULT_PATH = unique_named(
    "F30K_T3S3_INT8_NonInferiority_RESULT.json"
)
PER_IMAGE_PATH = unique_named(
    "F30K_T3S3_INT8_NonInferiority_PER_IMAGE.csv"
)
VALIDATION = read_json(VALIDATION_RESULT_PATH)
NONINFERIORITY = read_json(NONINFERIORITY_RESULT_PATH)

assert VALIDATION["validation_images"] == 1014
assert VALIDATION["test_set_status"] == "sealed"
assert VALIDATION["test_set_used"] is False
assert NONINFERIORITY["status"] == "passed"
assert NONINFERIORITY["test_set_status"] == "sealed"
assert NONINFERIORITY["test_set_used"] is False
assert NONINFERIORITY["source_result_sha256"] == sha256(
    VALIDATION_RESULT_PATH
)

captions = pd.read_csv(CAPTIONS_PATH)
per_image = pd.read_csv(PER_IMAGE_PATH)
data = captions.merge(
    per_image[["image_id", "fp32_CIDEr_D", "mixed_int8_CIDEr_D",
               "delta_CIDEr_D"]],
    on="image_id",
    how="inner",
    validate="one_to_one",
)
assert len(data) == 1014
assert int(data["exact_match"].sum()) == 767

needed_names = {f"{int(value)}.jpg" for value in data.image_id}
IMAGE_BY_NAME = {}
for path in INPUT_ROOT.rglob("*.jpg"):
    if path.name not in needed_names:
        continue
    old = IMAGE_BY_NAME.get(path.name)
    if old is None:
        IMAGE_BY_NAME[path.name] = path
    else:
        assert old.stat().st_size == path.stat().st_size
        IMAGE_BY_NAME[path.name] = min(
            [old, path],
            key=lambda item: (len(item.parts), len(str(item)), str(item)),
        )
assert set(IMAGE_BY_NAME) == needed_names

print("Authorization          : PASSED")
print("Validation images      :", len(data))
print("Changed captions       :", int((~data.exact_match).sum()))
print("Held-out Test          : SEALED / NOT READ")

Authorization          : PASSED
Validation images      : 1014
Changed captions       : 247
Held-out Test          : SEALED / NOT READ


## Cell 2 — Deterministic case selection and figure

In [2]:
changed = data[~data["exact_match"]].copy()
exact = data[data["exact_match"]].copy()

selected_parts = []
for label, frame in [
    ("largest_positive_delta", changed.nlargest(2, "delta_CIDEr_D")),
    ("largest_negative_delta", changed.nsmallest(2, "delta_CIDEr_D")),
    ("changed_nearest_zero",
     changed.iloc[(changed["delta_CIDEr_D"].abs()).argsort()[:2]]),
    ("exact_match_controls",
     pd.concat([
         exact.nlargest(1, "fp32_CIDEr_D"),
         exact.nsmallest(1, "fp32_CIDEr_D"),
     ])),
]:
    block = frame.copy()
    block["selection_group"] = label
    selected_parts.append(block)

selected = pd.concat(selected_parts, ignore_index=True)
selected = selected.drop_duplicates("image_id", keep="first")
assert len(selected) == 8, selected[["image_id", "selection_group"]]
selected_path = WORK_ROOT / f"{EXPERIMENT_ID}_CASES.csv"
selected.to_csv(selected_path, index=False, encoding="utf-8")

fig, axes = plt.subplots(4, 2, figsize=(16, 22))
for axis, row in zip(axes.flat, selected.itertuples(index=False)):
    with Image.open(IMAGE_BY_NAME[f"{int(row.image_id)}.jpg"]) as opened:
        image = opened.convert("RGB").copy()
    axis.imshow(image)
    axis.axis("off")
    title = (
        f"{row.selection_group} | image {int(row.image_id)} | "
        f"ΔCIDEr-D={row.delta_CIDEr_D:+.4f}\n"
        f"FP32: {row.fp32_caption}\n"
        f"INT8: {row.mixed_int8_caption}\n"
        f"Ref: {row.reference_1}"
    )
    axis.set_title("\n".join(textwrap.wrap(title, width=76)),
                   fontsize=9, loc="left")
fig.suptitle(
    "Flickr30k T3→S3: ONNX FP32 versus mixed INT8 — Validation cases",
    fontsize=15,
)
fig.tight_layout(rect=[0, 0, 1, 0.98])
figure_path = WORK_ROOT / f"{EXPERIMENT_ID}_CASES.png"
fig.savefig(figure_path, dpi=220, bbox_inches="tight")
plt.close(fig)
assert figure_path.stat().st_size > 100_000

print("Selected cases         :", len(selected))
print("Figure                 :", figure_path)

Selected cases         : 8
Figure                 : /kaggle/working/F30K_T3S3_INT8_Qualitative/F30K_T3S3_INT8_Qualitative_CASES.png


## Cell 3 — Descriptive result and manifest

In [3]:
result = {
    "schema_version": "f30k_t3s3_int8_qualitative_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete",
    "experiment_id": EXPERIMENT_ID,
    "evaluation_split": "Flickr30k Karpathy Validation",
    "selection_policy": {
        "largest_positive_delta": 2,
        "largest_negative_delta": 2,
        "changed_nearest_zero": 2,
        "exact_match_controls": 2,
        "selection_is_descriptive_not_confirmatory": True,
    },
    "selected_image_ids": [int(value) for value in selected.image_id],
    "source_integrity": {
        "validation_result_sha256": sha256(VALIDATION_RESULT_PATH),
        "captions_sha256": sha256(CAPTIONS_PATH),
        "noninferiority_result_sha256": sha256(NONINFERIORITY_RESULT_PATH),
        "per_image_sha256": sha256(PER_IMAGE_PATH),
    },
    "held_out_test_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json_atomic(result_path, result)
manifest_files = [selected_path, figure_path, result_path]
write_json_atomic(WORK_ROOT / "MANIFEST.json", {
    "schema_version": "f30k_t3s3_int8_qualitative_manifest_v1",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": [
        {"relative_path": path.name, "size_bytes": path.stat().st_size,
         "sha256": sha256(path)}
        for path in manifest_files
    ],
    "test_set_status": "sealed",
    "test_set_used": False,
})
print("Status                 : COMPLETE")
print("Interpretation         : DESCRIPTIVE ONLY")
print("Held-out Test          : SEALED / NOT USED")
print("Output                 :", WORK_ROOT)

Status                 : COMPLETE
Interpretation         : DESCRIPTIVE ONLY
Held-out Test          : SEALED / NOT USED
Output                 : /kaggle/working/F30K_T3S3_INT8_Qualitative
